# Tratamento e Padronização dos Dados(Use os dados da pasta dada/unfiltered)

In [17]:
import pandas as pd

# caminho do arquivo
caminho_csv = "Cultural and entertainment tourism Digital Demand, 0-100 (best)/aracaju_unfiltered.csv"

# carregar como DataFrame
df = pd.read_csv(caminho_csv, low_memory=False)

# visualizar
df.head()

,Cod_UF,regiao,id_vigitel,Cod_setor,Cod_municipio,Nome_do_municipio,Nome_da_UF,Nome_Grande_Regiao,Situacao_setor,Tipo_setor,...,tem_setor,Cod_uf,regiao_str,capital,pcnt_lit_7plus,IBP_Vigintel,IBP_Vigitel_str,cluster,cluster_hier_str_reordenado,IBP_Vigintel_ajustado
0,SE,2.0,258978.0,280030805000001,2800308.0,ARACAJU,Sergipe,Região Nordeste,1,0,...,1.0,SE,Nordeste,1.0,0.119231,2.0,Medium,2.0,2-0,NaN
1,SE,2.0,258979.0,280030805000001,2800308.0,ARACAJU,Sergipe,Região Nordeste,1,0,...,1.0,SE,Nordeste,1.0,0.119231,2.0,Medium,2.0,2-0,NaN
2,SE,2.0,258980.0,280030805000001,2800308.0,ARACAJU,Sergipe,Região Nordeste,1,0,...,1.0,SE,Nordeste,1.0,0.119231,2.0,Medium,2.0,2-0,NaN
3,SE,2.0,258981.0,280030805000002,2800308.0,ARACAJU,Sergipe,Região Nordeste,1,0,...,1.0,SE,Nordeste,1.0,0.021709,1.0,Low,1.0,1-0,NaN
4,SE,2.0,258982.0,280030805000002,2800308.0,ARACAJU,Sergipe,Região Nordeste,1,0,...,1.0,SE,Nordeste,1.0,0.021709,1.0,Low,1.0,1-0,NaN


In [18]:
#Tratamento / Merge de Colunas com Dados que se completam
# garantir numerico
cols = ["cozidadi", "cozidadia", "sofrutad", "sofrutadia"]
df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")

# criar colunas unificadas (prioriza a primeira, usa a segunda se estiver NaN)(As colunas se complementam)
df["cozidadi_cozidadia"] = df["cozidadi"].combine_first(df["cozidadia"])
df["sofrutad_sofrutadia"] = df["sofrutad"].combine_first(df["sofrutadia"])

# remover colunas antigas
df = df.drop(columns=["cozidadi", "cozidadia", "sofrutad", "sofrutadia"])

In [19]:
#Padronização dos dados
colunas = ["q7", "q47", "q70"]

# substituir 2.0 -> 0 
df[colunas] = df[colunas].replace(2.0, 0)

# tratamento especial q35(Valores inesperados/Erros)
df["q35"] = pd.to_numeric(df["q35"], errors="coerce")
df["q35"] = (df["q35"] == 1.0).astype(int)

In [20]:
df[colunas].value_counts(dropna=False)

q7   q47  q70
0.0  0    0      6152
     1    0      5623
1.0  1    0      4688
     0    0      2065
0.0  1    1       196
1.0  1    1       174
0.0  0    1       157
1.0  0    1        69
Name: count, dtype: int64

In [21]:
df["q35"].value_counts(dropna=False)

q35
0    12186
1     6938
Name: count, dtype: int64

In [22]:
#Conjunto de features atualmente usadas na geração de dados sinteticos
#E possivel adicionar outras mas provavelmente sera necessario um tratamento previo
selected_features = [
    'alcabu', 'q6', 'civil', 'q35', 'q47',
    'q70', 'adultos', 'af', 'q7', 'hortareg', 'sucodia', 'flvreco',
    'atiocu', 'freq', 'cruadia', 'atidom', 'frutareg', 'diab', 'atitrans',
    'tv_d_3', 'hart', 'saruim', 'sofrutad_sofrutadia', 'cozidadi_cozidadia',
    'q8_anos', 'excpeso_i', 'obesid_i', 'ativo_livre', 'pcnt_hh_inc_12',
    'pcnt_lit_7plus', 'av_No_water_swg_wc_grb_ex','cluster_hier_str_reordenado','cluster','ano'
]

target_features = ["fumante"]

# selecionar colunas
df_sub = df[selected_features + target_features]

In [23]:
df_sub

,alcabu,q6,civil,q35,q47,q70,adultos,af,q7,hortareg,...,excpeso_i,obesid_i,ativo_livre,pcnt_hh_inc_12,pcnt_lit_7plus,av_No_water_swg_wc_grb_ex,cluster_hier_str_reordenado,cluster,ano,fumante
0,0,70.0,2,0,0,0,2,1,1.0,0,...,0,0,1,15.000000,0.119231,0.000000,2-0,2.0,2010.0,0
1,0,68.0,3,0,0,0,1,0,0.0,0,...,1,0,0,15.000000,0.119231,0.000000,2-0,2.0,2013.0,0
2,1,48.0,2,1,1,0,3,1,1.0,1,...,1,1,0,15.000000,0.119231,0.000000,2-0,2.0,2016.0,0
3,0,64.0,3,0,0,0,2,1,0.0,0,...,,,1,4.385965,0.021709,0.000000,1-0,1.0,2006.0,0
4,0,41.0,2,1,1,0,2,2,1.0,1,...,,,0,4.385965,0.021709,0.000000,1-0,1.0,2007.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19119,0,55.0,2,1,1,0,3,1,1.0,1,...,0,0,0,26.146789,0.139441,0.540025,2-1,2.0,2016.0,0
19120,0,63.0,3,1,0,0,4,0,0.0,1,...,1,1,0,26.146789,0.139441,0.540025,2-1,2.0,2017.0,0
19121,0,19.0,1,0,0,0,4,1,1.0,0,...,0,0,1,26.146789,0.139441,0.540025,2-1,2.0,2017.0,0
19122,0,50.0,1,0,1,0,6,1,0.0,1,...,0,0,1,26.146789,0.139441,0.540025,2-1,2.0,2018.0,0


In [24]:
#Exportação dos dados filtrados
import os

pasta = "data_filtered"
os.makedirs(pasta, exist_ok=True)

# nome do arquivo
caminho_saida = os.path.join(pasta, "dados_filtrados.csv")

# salvar
df_sub.to_csv(caminho_saida, index=False)

# Geração dos Folds pra validação cruzada no pipeline

In [1]:
from pathlib import Path
import numpy as np
from sklearn.model_selection import StratifiedKFold, KFold
import pandas as pd
import pickle
import random

# Helpers

In [2]:
def load_samples(dataset_dir):
    with open(f"{dataset_dir}samples.pkl", "rb") as samples_file:
        return pickle.load(samples_file)

# Folds

In [3]:
import random
import numpy as np
from sklearn.model_selection import KFold, StratifiedKFold

def train_val_split(ids, split_indice=.9):
    # embaralha os índices
    random.shuffle(ids)
    
    # divide em treino e validação (ex: 90% treino, 10% val)
    return np.split(
        ids,
        [int(split_indice * len(ids))]
    )


def k_fold_split(ids, n_splits=5, random_state=72, shuffle=True):
    # lista que armazenará os folds
    folds = []

    # inicializa K-Fold
    kf = KFold(n_splits=n_splits, random_state=random_state, shuffle=shuffle)
    
    # percorre cada fold
    for fold_idx, splits in enumerate(kf.split(ids)):
        # splits[0] = índices de treino, splits[1] = índices de teste
        
        # separa treino em treino + validação
        train_ids, val_ids = train_val_split(splits[0].tolist())
        
        # define teste
        test_ids = splits[1].tolist()

        # armazena estrutura do fold
        folds.append({
            "fold_idx": fold_idx,
            "splits": {
                "train": train_ids.tolist(),
                "val": val_ids.tolist(),
                "test": test_ids
            }
        })

    return folds


def sk_fold_split(ids, cls, n_splits=5, random_state=72, shuffle=True):
    # lista que armazenará os folds
    folds = []

    # inicializa Stratified K-Fold (mantém proporção das classes)
    kf = StratifiedKFold(n_splits=n_splits, random_state=random_state, shuffle=shuffle)
    
    # percorre cada fold
    for fold_idx, splits in enumerate(kf.split(ids, cls)):
        # splits[0] = índices de treino, splits[1] = índices de teste
        
        # separa treino em treino + validação
        train_ids, val_ids = train_val_split(splits[0].tolist())
        
        # define teste
        test_ids = splits[1].tolist()

        # mapeia índices para os ids reais
        folds.append({
            "fold_idx": fold_idx,
            "splits": {
                "train": [int(ids[idx]) for idx in train_ids],
                "val": [int(ids[idx]) for idx in val_ids],
                "test": [int(ids[idx]) for idx in test_ids]
            }
        })

    return folds


def show_folds(folds, samples_to_show=10):
    # imprime resumo de cada fold
    for fold in folds:
        splits = fold['splits']
        
        print(f"fold: {fold['fold_idx']}")
        print(f"train({len(splits['train'])}): {splits['train'][:samples_to_show]}")
        print(f"val({len(splits['val'])}): {splits['val'][:samples_to_show]}")
        print(f"test({len(splits['test'])}): {splits['test'][:samples_to_show]}\n")

### stats

In [4]:
import numpy as np
import scipy.stats

def mean_confidence_interval(data, confidence=0.95):
    # converte os dados para array numpy
    a = 1.0 * np.array(data)
    
    # número de amostras
    n = len(a)
    
    # média e erro padrão da média (SEM)
    m, se = np.mean(a), scipy.stats.sem(a)
    
    # calcula a margem de erro usando distribuição t
    h = se * scipy.stats.t.ppf((1 + confidence) / 2., n - 1)
    
    # retorna no formato: média (intervalo), em porcentagem
    return f"{round(100 * m, 1)}({round(100 * h, 2)})"


def ic(result_df, metrics):
    # imprime o dataframe de resultados
    print(result_df)
    
    # calcula e imprime IC para cada métrica
    for metric in metrics:
        print(f"{metric}: {mean_confidence_interval(result_df[metric])}")

### checkpoint

In [5]:
def checkpoint_samples(samples_df, dataset_dir):
    with open(f"{dataset_dir}samples.pkl", "wb") as samples_file:
        pickle.dump(samples_df.to_dict(orient="records"), samples_file)

In [6]:
def checkpoint_folds(folds, dataset_dir):
    for fold in folds:
        fold_dir = f"{dataset_dir}fold_{fold['fold_idx']}/"
        Path(fold_dir).mkdir(parents=True, exist_ok=True)
        for split, split_ids in fold['splits'].items():
            with open(f"{fold_dir}{split}.pkl", "wb") as split_file:
                pickle.dump(split_ids,split_file)

# Vigitel V05 GM

In [7]:
# Nome do dataset que iremos salvar
dataset="Vigitel_V05_Teresina"

In [8]:
raw_samples_df = pd.read_csv("data_filtered/teresina_filtered.csv")
raw_samples_df.columns

Index(['alcabu', 'q6', 'civil', 'q35', 'q47', 'q70', 'adultos', 'af', 'q7',
       'hortareg', 'sucodia', 'flvreco', 'atiocu', 'freq', 'cruadia', 'atidom',
       'frutareg', 'diab', 'atitrans', 'tv_d_3', 'hart', 'saruim',
       'cozidadi_cozidadia', 'sofrutad_sofrutadia', 'q8_anos', 'excpeso_i',
       'obesid_i', 'ativo_livre', 'pcnt_hh_inc_12', 'pcnt_lit_7plus',
       'av_No_water_swg_wc_grb_ex', 'cluster_hier_str_reordenado', 'cluster',
       'ano', 'fumante'],
      dtype='object')

In [9]:
# Escolha de features que iremos usar na geração desses folds
selected_features = [
       'alcabu', 'q6', 'civil', 'q35', 'q47',
       'q70', 'adultos', 'af', 'q7', 'hortareg', 'sucodia', 'flvreco',
       'atiocu', 'freq', 'cruadia', 'atidom', 'frutareg', 'diab', 'atitrans',
       'tv_d_3', 'hart', 'saruim', 'cozidadi_cozidadia', 'sofrutad_sofrutadia',
       'q8_anos', 'excpeso_i', 'obesid_i', 'ativo_livre', 'pcnt_hh_inc_12',
       'pcnt_lit_7plus', 'av_No_water_swg_wc_grb_ex','cluster_hier_str_reordenado','cluster','ano'
]
target_features = ["fumante"]
len(selected_features + target_features)

35

In [10]:
samples_df = raw_samples_df.reset_index(drop=True)
samples_df["idx"] = samples_df.index
samples_df = samples_df
samples_df

,alcabu,q6,civil,q35,q47,q70,adultos,af,q7,hortareg,...,obesid_i,ativo_livre,pcnt_hh_inc_12,pcnt_lit_7plus,av_No_water_swg_wc_grb_ex,cluster_hier_str_reordenado,cluster,ano,fumante,idx
0,0,26.0,1,1,1,0,4,1,1.0,0,...,,0,64.197531,0.532656,46.580547,4-1,4.0,2006.0,1,0
1,0,49.0,2,1,1,0,4,0,1.0,0,...,,0,64.197531,0.532656,46.580547,4-1,4.0,2006.0,1,1
2,0,18.0,1,0,0,0,4,1,0.0,0,...,0,1,83.544304,0.575352,44.156347,4-1,4.0,2013.0,0,2
3,0,19.0,1,0,0,0,2,0,0.0,0,...,,0,56.983240,0.470335,41.239316,4-0,4.0,2006.0,0,3
4,1,54.0,1,1,0,0,1,0,1.0,0,...,,0,56.983240,0.470335,41.239316,4-0,4.0,2006.0,1,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19840,0,47.0,2,0,1,0,3,0,0.0,1,...,1,0,56.737589,0.182247,11.564781,3-2,3.0,2010.0,0,19840
19841,0,36.0,2,0,1,0,3,0,0.0,0,...,0,0,56.737589,0.182247,11.564781,3-2,3.0,2012.0,0,19841
19842,0,35.0,1,1,1,0,2,0,1.0,0,...,0,0,56.737589,0.182247,11.564781,3-2,3.0,2013.0,0,19842
19843,0,43.0,3,0,1,0,2,0,0.0,0,...,0,0,31.617647,0.086945,25.609756,3-1,3.0,2010.0,1,19843


### folds

In [11]:
samples_folds = sk_fold_split(ids=samples_df["idx"], cls=samples_df["fumante"])
show_folds(samples_folds, samples_to_show=16)

fold: 0
train(14288): [15964, 526, 2129, 6759, 18962, 12537, 8333, 11175, 7875, 2306, 14647, 16602, 17298, 4932, 8024, 14287]
val(1588): [8189, 1533, 7194, 2252, 6427, 19706, 18843, 18366, 2071, 13761, 10482, 6537, 4942, 5748, 9421, 18370]
test(3969): [1, 2, 3, 7, 11, 20, 21, 23, 31, 32, 36, 37, 51, 58, 73, 77]

fold: 1
train(14288): [3313, 18143, 8298, 18305, 12438, 17571, 4287, 12987, 14820, 2355, 1018, 12192, 14239, 9729, 183, 9856]
val(1588): [13376, 18939, 4723, 10861, 10066, 19109, 5666, 5769, 17053, 4793, 14902, 8601, 2837, 6217, 8643, 8563]
test(3969): [0, 4, 8, 10, 19, 28, 33, 34, 38, 42, 47, 48, 49, 52, 64, 68]

fold: 2
train(14288): [12670, 7438, 11999, 6869, 10338, 14829, 13969, 2776, 11317, 8347, 4547, 16695, 15997, 7775, 3561, 18138]
val(1588): [18852, 18467, 6136, 2778, 6854, 10142, 4830, 17013, 18236, 6833, 3530, 14916, 565, 15590, 5738, 6938]
test(3969): [5, 13, 14, 18, 22, 26, 29, 35, 43, 45, 46, 54, 55, 56, 57, 62]

fold: 3
train(14288): [13520, 18994, 8403, 4131, 91

In [12]:
dataset_dir=f"dataset/{dataset}/"
Path(dataset_dir).mkdir(parents=True, exist_ok=True)
checkpoint_samples(samples_df, dataset_dir)
checkpoint_folds(samples_folds, dataset_dir)